✅ Step 1: Load & Inspect Data

In [32]:
import pandas as pd
import numpy as np


# Load the dataset
df = pd.read_csv("data/expense_data_2.csv")

# Inspect the data
df.head()



,Income,Age,Dependents,Occupation,City_Tier,Rent,Loan_Repayment,Insurance,Groceries,Transport,...,Desired_Savings,Disposable_Income,Potential_Savings_Groceries,Potential_Savings_Transport,Potential_Savings_Eating_Out,Potential_Savings_Entertainment,Potential_Savings_Utilities,Potential_Savings_Healthcare,Potential_Savings_Education,Potential_Savings_Miscellaneous
0,44637.249636,49,0,Self_Employed,Tier_1,13391.174891,0.000000,2206.490129,6658.768341,2636.970696,...,6200.537192,11265.627707,1685.696222,328.895281,465.769172,195.151320,678.292859,67.682471,0.000000,85.735517
1,26858.596592,34,2,Retired,Tier_2,5371.719318,0.000000,869.522617,2818.444460,1543.018778,...,1923.176434,9676.818733,540.306561,119.347139,141.866089,234.131168,286.668408,6.603212,56.306874,97.388606
2,50367.605084,35,1,Student,Tier_3,7555.140763,4612.103386,2201.800050,6313.222081,3221.396403,...,7050.360422,13891.450624,1466.073984,473.549752,410.857129,459.965256,488.383423,7.290892,106.653597,138.542422
3,101455.600247,21,0,Self_Employed,Tier_3,15218.340037,6809.441427,4889.418087,14690.149363,7106.130005,...,16694.965136,31617.953615,1875.932770,762.020789,1241.017448,320.190594,1389.815033,193.502754,0.000000,296.041183
4,24875.283548,52,4,Professional,Tier_2,4975.056710,3112.609398,635.907170,3034.329665,1276.155163,...,1874.099434,6265.700532,788.953124,68.160766,61.712505,187.173750,194.117130,47.294591,67.388120,96.557076


✅ Step 2: Preprocess the Data

In [33]:
# Drop columns not needed for expense prediction from income
df = df.drop(columns=["Disposable_Income"], errors='ignore')

# Fill missing values
numeric_cols = df.select_dtypes(include=['number']).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


✅ Step 3: Split Features (X) and Labels (y)

In [34]:
# Define expense columns to be predicted
expense_columns = ["Rent", "Loan_Repayment", "Insurance", "Groceries", "Transport",
                   "Eating_Out", "Entertainment", "Utilities", "Healthcare", "Education", "Miscellaneous"]

# Feature: Only income
X = df[["Income"]]

# Targets: Expenses
y = df[expense_columns]


✅ Step 4: Train-Test Split

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


✅ Step 5: Train the Model (XGBoost)

In [36]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42
)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

✅ Step 6: Evaluate Performance

In [37]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred, multioutput='raw_values')
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')

for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2[i]:.4f}, MAE = {mae[i]:.2f}")


Rent: R² = 0.8209, MAE = 2027.37
Loan_Repayment: R² = 0.1815, MAE = 2477.01
Insurance: R² = 0.8128, MAE = 322.06
Groceries: R² = 0.9066, MAE = 547.27
Transport: R² = 0.8458, MAE = 345.49
Eating_Out: R² = 0.8786, MAE = 305.10
Entertainment: R² = 0.8060, MAE = 320.79
Utilities: R² = 0.8898, MAE = 424.27
Healthcare: R² = 0.9083, MAE = 216.20
Education: R² = 0.5981, MAE = 1090.21
Miscellaneous: R² = 0.8312, MAE = 205.68


✅ Step 7: Save the Model 

In [38]:
import joblib
joblib.dump(model, "income_to_expense_model.pkl")


['income_to_expense_model.pkl']

✅ Step 8: Feature Engineering

In [44]:
# ✅ Create new income-related features


df["Income_Log"] = np.log1p(df["Income"])
df["Income_Square"] = df["Income"] ** 2
df["Income_Bucket"] = pd.qcut(df["Income"], q=4, labels=[1, 2, 3, 4]).astype(int)


✅ Step 9: Retrain with New Features 

In [45]:
# Use the new features
feature_cols = ["Income", "Income_Log", "Income_Square", "Income_Bucket"]
X = df[feature_cols]

# Split again
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train again with XGB
model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.5,
    reg_lambda=0.5,
    random_state=42
)
model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

✅ Step 10: Re-Evaluate

In [46]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred, multioutput='raw_values')
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')

print("🔍 Model Evaluation After Feature Engineering:")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2[i]:.4f}, MAE = {mae[i]:.2f}")


🔍 Model Evaluation After Feature Engineering:
Rent: R² = 0.8211, MAE = 2026.97
Loan_Repayment: R² = 0.1809, MAE = 2477.66
Insurance: R² = 0.8133, MAE = 322.09
Groceries: R² = 0.9061, MAE = 546.95
Transport: R² = 0.8453, MAE = 345.49
Eating_Out: R² = 0.8789, MAE = 304.83
Entertainment: R² = 0.8057, MAE = 320.95
Utilities: R² = 0.8892, MAE = 424.42
Healthcare: R² = 0.9083, MAE = 216.13
Education: R² = 0.5980, MAE = 1090.92
Miscellaneous: R² = 0.8313, MAE = 205.81


✅ Step 12: Add Grid Search Cell

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

param_grid = {
    'estimator__n_estimators': [200, 300, 400],
    'estimator__learning_rate': [0.01, 0.03, 0.05],
    'estimator__max_depth': [3, 4, 5],
    'estimator__subsample': [0.7, 0.8, 0.9],
    'estimator__colsample_bytree': [0.7, 0.8, 0.9],
    'estimator__reg_alpha': [0.1, 0.5, 1.0],
    'estimator__reg_lambda': [0.1, 0.5, 1.0],
}

# Grid search with multi-output wrapper
from sklearn.multioutput import MultiOutputRegressor
xgb = XGBRegressor(random_state=42, verbosity=0)
grid = GridSearchCV(
    estimator=MultiOutputRegressor(xgb),
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)


Fitting 3 folds for each of 2187 candidates, totalling 6561 fits


GridSearchCV(cv=3,
             estimator=MultiOutputRegressor(estimator=XGBRegressor(base_score=None,
                                                                   booster=None,
                                                                   callbacks=None,
                                                                   colsample_bylevel=None,
                                                                   colsample_bynode=None,
                                                                   colsample_bytree=None,
                                                                   device=None,
                                                                   early_stopping_rounds=None,
                                                                   enable_categorical=False,
                                                                   eval_metric=None,
                                                                   feature_types=None,
                                                                   gamma=None,
                                                                   grow_policy=None,
                                                                   importance_type=None,
                                                                   interaction_constr...
                                                                   num_parallel_tree=None,
                                                                   random_state=42, ...)),
             n_jobs=-1,
             param_grid={'estimator__colsample_bytree': [0.7, 0.8, 0.9],
                         'estimator__learning_rate': [0.01, 0.03, 0.05],
                         'estimator__max_depth': [3, 4, 5],
                         'estimator__n_estimators': [200, 300, 400],
                         'estimator__reg_alpha': [0.1, 0.5, 1.0],
                         'estimator__reg_lambda': [0.1, 0.5, 1.0],
                         'estimator__subsample': [0.7, 0.8, 0.9]},
             scoring='r2', verbose=2)

✅ Step 13: Use Best Model and Evaluate

In [50]:
# Get the best model from grid search
best_model = grid.best_estimator_

# Predict and evaluate
y_pred_best = best_model.predict(X_test)

r2_best = r2_score(y_test, y_pred_best, multioutput='raw_values')
mae_best = mean_absolute_error(y_test, y_pred_best, multioutput='raw_values')

print("📊 Best Model Evaluation (Grid Search):")
for i, col in enumerate(expense_columns):
    print(f"{col}: R² = {r2_best[i]:.4f}, MAE = {mae_best[i]:.2f}")


📊 Best Model Evaluation (Grid Search):
Rent: R² = 0.8206, MAE = 2030.19
Loan_Repayment: R² = 0.1847, MAE = 2473.41
Insurance: R² = 0.8130, MAE = 321.90
Groceries: R² = 0.9059, MAE = 548.25
Transport: R² = 0.8450, MAE = 345.80
Eating_Out: R² = 0.8789, MAE = 305.24
Entertainment: R² = 0.8063, MAE = 320.45
Utilities: R² = 0.8896, MAE = 424.38
Healthcare: R² = 0.9081, MAE = 216.49
Education: R² = 0.5988, MAE = 1088.89
Miscellaneous: R² = 0.8306, MAE = 205.82


✅ Step 11: Save Updated Mode